# Titanic Survival Prediction

## Problem
Build a machine learning model to predict whether a passenger survived the Titanic disaster.

## Target
Survived

## Goal
Build an end-to-end classification pipeline, compare multiple models, and evaluate the final model on unseen data.

In [1]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml(
    name='titanic',
    version=1,
    as_frame=True
)

df = titanic.frame
df.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [2]:
df.shape

(1309, 14)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   str     
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   str     
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    str     
 10  embarked   1307 non-null   category
 11  boat       486 non-null    str     
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    str     
dtypes: category(3), float64(3), int64(3), str(5)
memory usage: 116.5 KB


In [4]:
df.isna().sum()

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

In [5]:
df['survived'].value_counts()

survived
0    809
1    500
Name: count, dtype: int64

In [6]:
df['survived'].value_counts(normalize=True)

survived
0    0.618029
1    0.381971
Name: proportion, dtype: float64

In [7]:
features = [
    'pclass',
    'sex',
    'age',
    'sibsp',
    'parch',
    'fare',
    'embarked'
]

X = df[features]
y = df['survived']

In [8]:
numeric_features = ['pclass', 'age', 'sibsp', 'parch', 'fare']
categorical_features = ['sex', 'embarked']

In [9]:
X.isna().sum()

pclass        0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2
dtype: int64

In [10]:
(X.isna().mean() * 100).round(2)

pclass       0.00
sex          0.00
age         20.09
sibsp        0.00
parch        0.00
fare         0.08
embarked     0.15
dtype: float64

## Missing Values
- Age contains 20.09% of missing values.
- Fare contains 0.08%.
- Embarked contains 0.15%.

Numerical missing values will be imputed using the median.
Categorical missing values will be imputed using the most frequent value.

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

The final test set will remain untouched until model selection and hyperparameter tuning are complete.

## Modeling Plan
1. Split the data into training and final test sets.
2. Build preprocessing pipelines for numerical and categorical features.
3. Compare Logistic Regression and Random Forest using cross-validation.
4. Select an evaluation metric.
5. Tune the selected model with GridSearchCV.
6. Evaluate the final model once on the untouched test set.
7. Analyze errors and interpret the results.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

In [14]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [15]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [16]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ]
)

In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

In [18]:
from sklearn.model_selection import cross_validate

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [19]:
y_train.value_counts()

survived
0    647
1    400
Name: count, dtype: int64

In [21]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [22]:
logistic_cv = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=scoring
)

In [23]:
rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=scoring
)

In [24]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        logistic_cv["test_accuracy"].mean(),
        rf_cv["test_accuracy"].mean()
    ],
    "Precision": [
        logistic_cv["test_precision"].mean(),
        rf_cv["test_precision"].mean()
    ],
    "Recall": [
        logistic_cv["test_recall"].mean(),
        rf_cv["test_recall"].mean()
    ],
    "F1": [
        logistic_cv["test_f1"].mean(),
        rf_cv["test_f1"].mean()
    ]
})

comparison.round(3)

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.777,0.719,0.688,0.702
1,Random Forest,0.777,0.723,0.678,0.698


In [26]:
print(
    "Logistic F1:",
    logistic_cv["test_f1"].mean().round(5),
    "+/-",
    logistic_cv["test_f1"].std().round(5)
)

print(
    "Random Forest F1:",
    rf_cv["test_f1"].mean().round(5),
    "+/-",
    rf_cv["test_f1"].std().round(5)
)

Logistic F1: 0.70157 +/- 0.02376
Random Forest F1: 0.698 +/- 0.052
